# New Inference

In [2]:
import torch
from PIL import Image
import argparse
import os, json, random
import pandas as pd
import matplotlib.pyplot as plt
import glob, re

# from tqdm.notebook import tqdm
from tqdm import tqdm
import numpy as np

from safetensors.torch import load_file
import matplotlib.image as mpimg
import copy
import gc
from transformers import CLIPTextModel, CLIPTokenizer
from safetensors.torch import load_file

import diffusers
from diffusers import DiffusionPipeline
from diffusers import AutoencoderKL, DDPMScheduler, DiffusionPipeline, UNet2DConditionModel, LMSDiscreteScheduler
from diffusers.loaders import AttnProcsLayers
from diffusers.models.attention_processor import LoRAAttnProcessor, AttentionProcessor
from typing import Any, Dict, List, Optional, Tuple, Union
from trainscripts.textsliders.lora import LoRANetwork, DEFAULT_TARGET_REPLACE, UNET_TARGET_REPLACE_MODULE_CONV

/home/sjq/.conda/envs/sliders/lib/python3.9/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
def flush():
    torch.cuda.empty_cache()
    gc.collect()
flush()
width = 512
height = 512 
steps = 50  
cfg_scale = 7.5 
pretrained_sd_model = "CompVis/stable-diffusion-v1-4"

In [ ]:
pretrained_model_name_or_path = "CompVis/stable-diffusion-v1-4"

revision = None
device = 'cuda:3'
rank = 8
weight_dtype = torch.float16

# Load scheduler, tokenizer and models.
noise_scheduler = LMSDiscreteScheduler(beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear", num_train_timesteps=1000)
tokenizer = CLIPTokenizer.from_pretrained(
    pretrained_model_name_or_path, subfolder="tokenizer", revision=revision
)
text_encoder = CLIPTextModel.from_pretrained(
    pretrained_model_name_or_path, subfolder="text_encoder", revision=revision
)
vae = AutoencoderKL.from_pretrained(pretrained_model_name_or_path, subfolder="vae", revision=revision)
unet = UNet2DConditionModel.from_pretrained(
    pretrained_model_name_or_path, subfolder="unet", revision=revision
)
# freeze parameters of models to save more memory
unet.requires_grad_(False)
unet.to(device, dtype=weight_dtype)
vae.requires_grad_(False)

text_encoder.requires_grad_(False)

# For mixed precision training we cast all non-trainable weigths (vae, non-lora text_encoder and non-lora unet) to half-precision
# as these weights are only used for inference, keeping weights in full precision is not required.


# Move unet, vae and text_encoder to device and cast to weight_dtype
vae.requires_grad_(False)
vae.to(device, dtype=weight_dtype)
text_encoder.to(device, dtype=weight_dtype)

In [ ]:
# path to your model file
lora_weights = [

    # 'models/age_noxattn_gender_race_alpha4.0_rank4_noxattn/age_noxattn_gender_race_alpha4.0_rank4_noxattn_500steps.pt',
    # '/home/sjq/concept_sliders/models/tornadoslider_alpha1.0_rank8_noxattn/tornadoslider_alpha1.0_rank8_noxattn_last.safetensors'
    # '/home/sjq/concept_sliders/models/socalfire_alpha1.0_rank8_noxattn/socalfire_alpha1.0_rank8_noxattn_last.safetensors',
    '/home/sjq/concept_sliders/models/wildfire_alpha1.0_rank8_noxattn/wildfire_alpha1.0_rank8_noxattn_last.safetensors',
    
]

In [ ]:
# prompts to try
prompts = [ 
            # "a high-resolution satellite image, overhead view of a coastal urban area, residential buildings, roads, vegetation, clear weather, natural colors",
            # "a high-resolution satellite image, overhead view",
            # 'high-resolution satellite remote sensing image, true overhead orthographic view, real-world urban area, natural terrain texture, realistic building rooftops, detailed road networks, subtle shadows consistent with sun angle, atmospheric scattering, sensor noise, slight haze, real satellite color calibration, unedited raw satellite photo, geographic accuracy, non-artistic, non-stylized, Earth observation imagery',
            # 'high resolution satellite image of Santa Rosa, California, true top-down view, realistic earth observation imagery, natural colors, suburban neighborhoods, vineyards, dry hills, sharp details, photorealistic',
            # 'high resolution satellite image of Santa Rosa, California, true top-down view, realistic earth observation imagery, natural colors, suburban neighborhoods, vineyards, dry hills, sharp details, photorealistic',
            'high resolution satellite image of Santa Rosa, California before wildfire, true top-down orthographic view, natural color RGB, suburban neighborhoods, vineyards, dry hills, intact vegetation, clear road network, realistic earth observation imagery, sharp details, photorealistic',
            'satellite imagery of Santa Rosa, California before disaster, orthographic top-down view, natural vegetation, no smoke, no fire, no damage, suburban residential area and vineyards, realistic earth observation style, high detail',
          ]

# LoRA weights/scale to test
scales = [0, 0.3, 0.5, 0.8, 1, 1.2, 1.5]

# timestep during inference when we switch to LoRA scale>0 (this is done to ensure structure in the images)
# start_noise = 800


#number of images per prompt
num_images_per_prompt = 1

torch_device = device
negative_prompt = None
batch_size = 1
height = 512
width = 512
ddim_steps = 50
guidance_scale = 7.5



def load_lora_state(path: str):
    if path.endswith(".safetensors"):
        sd = load_file(path, device="cpu")
    else:
        sd = torch.load(path, map_location="cpu")
    if isinstance(sd, dict) and "state_dict" in sd and isinstance(sd["state_dict"], dict):
        sd = sd["state_dict"]
    return sd


for prompt in prompts:
    # for different seeds on same prompt
    for _ in range(num_images_per_prompt):
        seed = random.randint(0, 5000)
        for lora_weight in lora_weights:
        
            if 'full' in lora_weight:
                train_method = 'full'
            elif 'noxattn' in lora_weight:
                train_method = 'noxattn'
            else:
                train_method = 'noxattn'

            network_type = "c3lier"
            if train_method == 'xattn':
                network_type = 'lierla'

            modules = DEFAULT_TARGET_REPLACE
            if network_type == "c3lier":
                modules += UNET_TARGET_REPLACE_MODULE_CONV
            import os
            model_name = lora_weight

            name = os.path.basename(model_name)
            unet = UNet2DConditionModel.from_pretrained(
                pretrained_model_name_or_path, subfolder="unet", revision=revision
            )
            # freeze parameters of models to save more memory
            unet.requires_grad_(False)
            unet.to(device, dtype=weight_dtype)
            rank = 16
            alpha = 1
            if 'rank4' in lora_weight:
                rank = 4
            if 'rank8' in lora_weight:
                rank = 8
            if 'alpha1' in lora_weight:
                alpha = 1.0
            network = LoRANetwork(
                    unet,
                    rank=rank,
                    multiplier=1.0,
                    alpha=alpha,
                    train_method=train_method,
                ).to(device, dtype=weight_dtype)
            # network.load_state_dict(torch.load(lora_weight))
            sd = load_lora_state(lora_weight)
            missing, unexpected = network.load_state_dict(sd, strict=False)
            print("missing:", len(missing), "unexpected:", len(unexpected))

            images_list = []

            print(prompt, seed)

            for scale in scales:

                generator = torch.manual_seed(seed) 
                text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")

                text_embeddings = text_encoder(text_input.input_ids.to(torch_device))[0]

                max_length = text_input.input_ids.shape[-1]
                if negative_prompt is None:
                    uncond_input = tokenizer(
                        [""] * batch_size, padding="max_length", max_length=max_length, return_tensors="pt"
                    )
                else:
                    uncond_input = tokenizer(
                        [negative_prompt] * batch_size, padding="max_length", max_length=max_length, return_tensors="pt"
                    )
                uncond_embeddings = text_encoder(uncond_input.input_ids.to(torch_device))[0]

                text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

                latents = torch.randn(
                    (batch_size, unet.in_channels, height // 8, width // 8),
                    generator=generator,
                )
                latents = latents.to(torch_device)

                noise_scheduler.set_timesteps(ddim_steps)

                latents = latents * noise_scheduler.init_noise_sigma
                latents = latents.to(weight_dtype)
                latent_model_input = torch.cat([latents] * 2)

                # add lora just in the last 70% part
                timesteps = list(noise_scheduler.timesteps)
                switch_i = int(len(timesteps) * 0.4)
                
                # for t in tqdm(noise_scheduler.timesteps):
                    # if t>start_noise:
                    #     network.set_lora_slider(scale=0)
                    # else:
                    #     network.set_lora_slider(scale=scale)
                
                for i, t in enumerate(tqdm(timesteps)):
                    network.set_lora_slider(scale=0 if i < switch_i else scale)
                    # expand the latents if we are doing classifier-free guidance to avoid doing two forward passes.
                    latent_model_input = torch.cat([latents] * 2)

                    latent_model_input = noise_scheduler.scale_model_input(latent_model_input, timestep=t)
                    # predict the noise residual
                    with network:
                        with torch.no_grad():
                            noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample
                    # perform guidance
                    noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
                    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

                    # compute the previous noisy sample x_t -> x_t-1
                    latents = noise_scheduler.step(noise_pred, t, latents).prev_sample

                # scale and decode the image latents with vae
                latents = 1 / 0.18215 * latents
                with torch.no_grad():
                    image = vae.decode(latents).sample
                image = (image / 2 + 0.5).clamp(0, 1)
                image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
                images = (image * 255).round().astype("uint8")
                pil_images = [Image.fromarray(image) for image in images]
                images_list.append(pil_images[0])
            del network, unet
            unet = None
            network = None
            torch.cuda.empty_cache()
            flush()
            fig, ax = plt.subplots(1, len(images_list), figsize=(20,4))
            for i, a in enumerate(ax):
                a.imshow(images_list[i])
                a.set_title(f"{scales[i]}",fontsize=15)
                a.axis('off')

#             plt.suptitle(f"{os.path.basename(lora_weight).replace('.pt','')}", fontsize=20)            

            # plt.tight_layout()
            plt.show()

NameError: name 'device' is not defined